An example that LLM sends back tool_use with multiple tool calls

disable_parallel_tool_use does the trick

```
response = client.messages.create(
    model="claude-opus-5-5",
    max_tokens=1024,
    tools=tools,
    # Ask for at most one tool call per turn.
    tool_choice={"type": "auto", "disable_parallel_tool_use": False},
    messages=messages,
)
```

In [ ]:
from util.claude_client import get_claude_client_and_default_model

client = get_claude_client_and_default_model('claude')





In [ ]:
tools = [
    {
        'name': 'get_time',
        'description' : 'find the current time in a given timezone',
        'input_schema': {
            'type': 'object',
            'properties': {
                'timezone': {
                    'type': 'string',
                    'description' : 'The timezone, e.g. EST' ,
                    'default': 'UTC'},
            },
            'required': ['timezone']

        }

    },

    {
        'name': 'get_weather',
        'description': 'Get the current weather in a given location',
        'input_schema': {
            'type': 'object',
            'properties': {
                'location': {
                    'type': 'string',
                    'description' : 'The city and state, e.g. San Francisco, CA',
                }
            }
        },
        'required': ['location']
    }

]
messages = [
    {
        "role": "user",
        "content": "What's the weather in SF and NYC, and what time is it there?",
    }
]


def call_tool( tool_name : str , input:object ):

    if tool_name == 'get_time':
        if "PDT" in str(tool_use.input):
            result = "2:30 PM PDT"
        elif "PST" in str(tool_use.input):
            result = "1:30 PM PDT"
        elif "EDT" in str(tool_use.input):
            result = "5:30 PM EDT"
        else:
            result = "4:30 PM EST"
    else:
        if "San Francisco" in str(tool_use.input):
            result = "San Francisco: 68°F, partly cloudy"
        else:
            result = "New York: 45°F, clear skies"
    return result


In [ ]:
client  , llm= get_claude_client_and_default_model('deepseek')
message = client.messages.create(
    max_tokens=1024,
    model=llm,
    messages = messages,
    tools=tools,
    tool_choice={ "type":"auto" , "disable_parallel_tool_use": False},
)

messages += [

    {
        "role": "assistant",
        "content": message.content
    }

]
while message.stop_reason != 'end_turn':

    print( message.model_dump_json(indent=4) )

    tool_uses = [block for block in message.content if block.type == 'tool_use']
    print(f"number of tools used: {len(tool_uses)}")
    for tool_use in tool_uses:
        tool_execution_result = call_tool(tool_use.name, tool_use.input)

        messages += [

            {
                'role': 'user',
                'content': {
                    'type':'tool_result',
                    'tool_use_id': tool_use.id,
                    'content': tool_execution_result
                 }
            }

        ]

    message = client.messages.create(
        max_tokens=1024,
        model=llm,
        messages = messages,
        tools=tools,
        tool_choice={"type":"auto" ,  "disable_parallel_tool_use": False},
    )

print( message.model_dump_json(indent=4) )
